In [ ]:
!pip install -q transformers sentence-transformers razdel

# Генерация train_augmentation.csv

In [ ]:
import re
import random
import warnings
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Device: cuda


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'
output_dir = os.path.join(drive_root, 'output')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(drive_root, 'train.csv')
out_path = os.path.join(output_dir, 'paraphrase-3')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
TARGET_PER_CLASS = 40
SIM_MIN = 0.80
SIM_MAX = 0.95
SIM_LABEL_MIN = 0.8
SIM_LABEL_MAX = 0.98

In [ ]:
df = pd.read_csv(train_path)
counts = df["label"].value_counts()
small_labels = counts[counts < TARGET_PER_CLASS].sort_values(ascending=True).index
df_small = df[df["label"].isin(small_labels)]

# Расчёт общего числа аугментаций
TOTAL_AUGMENTED = (TARGET_PER_CLASS * len(small_labels)) - len(df_small)

print(f"Всего малых классов: {len(small_labels)}")
print(f"Текущее количество примеров в малых классах: {len(df_small)}")
print(f"Целевое количество на класс: {TARGET_PER_CLASS}")
print(f"Нужно сгенерировать аугментаций: {TOTAL_AUGMENTED}")

Всего малых классов: 26
Текущее количество примеров в малых классах: 367
Целевое количество на класс: 40
Нужно сгенерировать аугментаций: 673


In [ ]:
source_text = "[ORGANIZATION] ПРИКАЗ [DATE_TIME] No [DOCUMENT_NUMBER] Об утверждении Положений по непрофильным активам и порядке отчуждения непрофильных активов В целях приведения локальных нормативных актов Группы [ORGANIZATION] по вопросам управления непрофильными активами в соответствии с нормативными актами [ORGANIZATION] ПРИКАЗЫВАЮ: 1. Утвердить Положение о Комиссии по непрофильным активам [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 1). 2. Утвердить Положение о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] в новой редакции (Приложение 2). 3. Утвердить состав Комиссии по непрофильным активам [ORGANIZATION] в новом составе (Приложение 3). 4. Признать утратившими силу приказы [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] и от [DATE_TIME] No [DOCUMENT_NUMBER]. 5. Распространить действие настоящего Приказа на организации Группы компаний [ORGANIZATION], в которых Компания имеет право прямо или косвенно распоряжаться более 50 голосов (Приложение 4). 6. Контроль исполнения приказа возложить на заместителя Председателя Правления [PERSON] Председатель Правления [PERSON] Приложение 1 к приказу [ORGANIZATION] от [DATE_TIME] No [DOCUMENT_NUMBER] ПОЛОЖЕНИЕ О КОМИССИИ [ORGANIZATION] И ОРГАНИЗАЦИЙ ГРУППЫ КОМПАНИЙ [ORGANIZATION] ПО НЕПРОФИЛЬНЫМ АКТИВАМ 1. Общие положения Настоящее Положение определяет цели, задачи, организацию деятельности Комиссии [ORGANIZATION] по непрофильным активам (далее – Комиссия), права и обязанности Председателя, Секретаря и членов Комиссии, а также порядок представления отчетов о ходе ее работы. Требования настоящего Положения обязательны для применения при отчуждении непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION]. Дочерние общества и организации [ORGANIZATION] обеспечивают утверждение локальных нормативных актов (положений), регламентирующих в соответствии с настоящим Положением порядок работы комиссий по непрофильным активам в дочерних обществах, а также в объектах вложений дочерних обществ, являющихся организациями Группы компаний [ORGANIZATION]. Представители интересов [ORGANIZATION] и дочерних обществ [ORGANIZATION] в органах управления организаций Группы компаний [ORGANIZATION] обеспечивают на основе настоящего Положения утверждение локальных нормативных актов (положений), регламентирующих порядок работы комиссий по непрофильным активам в таких организациях Группы компаний [ORGANIZATION]. 2. Основные цели и задачи Комиссии 2.1. Комиссия создается в целях обеспечения реализации Стратегии по управлению имуществом и иными активами [ORGANIZATION], Программы отчуждения непрофильных активов [ORGANIZATION], утверждаемой [ORGANIZATION], и Положения о комиссии [ORGANIZATION] по непрофильным активам. Комиссия в своей деятельности руководствуется законодательством Российской Федерации, решениями органов управления [ORGANIZATION] и [ORGANIZATION], иными внутренними документами, в том числе Положением о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы Газпром, Положением о комиссии [ORGANIZATION] по непрофильным активам, Положением о порядке отчуждения непрофильных активов [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] и настоящим Положением. 2.2. Основной задачей Комиссии является принятие решений об отнесении активов, принадлежащих [ORGANIZATION] и организациям Группы компаний [ORGANIZATION], к категории непрофильных, а также определение порядка их реализации. В отношении активов, принадлежащих [ORGANIZATION] и организациям Группы компаний [ORGANIZATION], в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет 100 (сто) и более миллионов рублей с учетом НДС решения принимает Комиссия [ORGANIZATION] по непрофильным активам. В отношении активов, принадлежащих [ORGANIZATION], а также организациям Группы компаний [ORGANIZATION] в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет от 30 до 100 миллионов рублей с учетом НДС, решения принимаются Комиссией [ORGANIZATION]. В отношении активов, принадлежащих организациям Группы компаний [ORGANIZATION], в случае, если их балансовая (остаточная) стоимость по данным бухгалтерского учета, или рыночная стоимость, или кадастровая стоимость (для земельных участков)1 составляет менее 30 миллионов рублей с учетом НДС, решения принимаются Комиссией организации Группы компаний [ORGANIZATION]. 2.3. Для выполнения возложенной на нее задачи, Комиссия определяет критерии отнесения активов к категории непрофильных, а также принимает следующие решения: об отнесении активов к категории непрофильных; о признании непрофильных активов подлежащими/не подлежащими отчуждению; о целесообразности применения тех или иных способов отчуждения непрофильных активов; об определении условий отчуждения непрофильных активов2; о необходимости проведения предпродажной подготовки непрофильных активов, в том числе оценки их рыночной стоимости. 3. Организация деятельности Комиссии 3.1. Персональный состав Комиссии утверждается приказом [ORGANIZATION]. 3.2. Вопросы об отнесении активов к категории непрофильных вносятся на рассмотрение Комиссии по инициативе органов управления [ORGANIZATION], заместителей Председателя Правления и руководителей подразделений прямого подчинения Председателю Правления [ORGANIZATION], структурных подразделений [ORGANIZATION]. 3.3. Организация Группы компаний [ORGANIZATION] перед представлением материалов на рассмотрение Комиссии согласовывает обоснование целесообразности отчуждения актива, а также предполагаемого способа его отчуждения со структурными подразделениями [ORGANIZATION], ответственными за осуществление контроля за обеспечением эффективности долгосрочных финансовых вложений [ORGANIZATION] в соответствующие дочерние общества [ORGANIZATION], которым подконтрольна эта организация Группы компаний [ORGANIZATION] (далее - ответственное подразделение [ORGANIZATION]). Представление материалов на рассмотрение Комиссии осуществляется дочерними обществами [ORGANIZATION], в том числе в отношении подконтрольных организаций Группы компаний [ORGANIZATION], являющихся их объектами вложений, либо ответственным подразделением [ORGANIZATION]. 3.4 Комиссия по вопросам, входящим в ее компетенцию, имеет право: запрашивать у структурных подразделений [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] необходимые для ее деятельности документы, материалы и информацию; устанавливать для структурных подразделений [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] сроки и форму представления 1 В отношении объектов имущества, имеющих одни и те же идентификационные признаки (место нахождения, назначение и другие признаки), а также отчуждаемых в пользу одного приобретателя либо входящих в единый имущественный комплекс, рассчитывается суммарная балансовая (остаточная) или рыночная или кадастровая стоимость имущества (для земельных участков). 2 Во избежание дублирования норм и правил работы Комиссии и Инвестиционных комитетов Компании, вопросы о целесообразности отчуждения непрофильных активов могут быть приняты к рассмотрению Комиссией только после их рассмотрения на соответствующем Инвестиционном Комитете запрашиваемых документов, материалов и информации. 3.5. Заседания Комиссии проводятся по мере необходимости. Заседание Комиссии является правомочным при участии в нем не менее половины от общего числа ее членов. При очной форме проведения заседания Комиссии решения принимаются открытым голосованием простым большинством голосов членов Комиссии, присутствующих на заседании. При заочной форме проведения заседания Комиссии голосование осуществляется посредством бюллетеней, которые направляются ее членам в соответствии с настоящим Положением. Бюллетени для голосования должны содержать указание на дату представления заполненного бюллетеня. В случае равенства голосов решающим является голос Председателя Комиссии, а в его отсутствие заместителя Председателя Комиссии. 3.6. Повестка заседания Комиссии, письмо о созыве заседания с указанием формы, даты, времени и места его проведения, подписанные Председателем Комиссии (в его отсутствие заместителем Председателя Комиссии), информационные материалы и бюллетени для голосования (при проведении заседания в заочной форме) направляются членам Комиссии за 5 календарных дней до даты проведения заседания Комиссии. В исключительных случаях при необходимости принятия оперативных решений указанный срок может быть сокращен по решению Председателя Комиссии (в его отсутствие заместителя Председателя Комиссии). 3.7. Итоги голосования членов Комиссии оформляются протоколом в течение 3 календарных дней с даты проведения заседания. Протокол подписывается Председателем Комиссии (в его отсутствие заместителем Председателя Комиссии) и Секретарем Комиссии. При принятии Комиссией решений заочным голосованием протокол заседания оформляется в течение 3 календарных дней с даты, установленной для представления заполненных бюллетеней. К протоколу прилагаются подписанные членами Комиссии бюллетени для голосования. Бюллетень члена Комиссии, не представленный Секретарю Комиссии в установленный для голосования срок, либо заполненный ненадлежащим образом, при подведении итогов голосования не учитывается. Члены Комиссии, голосовавшие против принятого решения, а также воздержавшиеся вправе в письменной форме изложить свое особое мнение, которое приобщается к протоколу заседания Комиссии. Протокол заседания Комиссии должен содержать информацию о членах Комиссии, проголосовавших по вопросам повестки заседания, результатах голосования, а также о принятых на заседании решениях. 3.8. Решение Комиссии (выписки из протокола заседания Комиссии) направляются для исполнения в соответствующее структурное подразделение или дочернее общество [ORGANIZATION] в течение 3 рабочих дней с даты оформления соответствующего протокола Комиссии. 4. Права и обязанности Председателя, Секретаря и членов Комиссии 4.1. Председателем Комиссии назначается руководитель структурного подразделения [ORGANIZATION], одной из основных задач которого является обеспечение управления непрофильными активами [ORGANIZATION] и организаций Группы компаний [ORGANIZATION] (далее Департамент по правовым, корпоративным и имущественным вопросам). В отсутствие Председателя Комиссии обязанности Председателя исполняет его заместитель. Председатель Комиссии: организует работу Комиссии и выполнение возложенных на нее задач; созывает, проводит заседания Комиссии и председательствует на них; обеспечивает коллегиальное обсуждение рассматриваемых вопросов; при необходимости дает поручения членам Комиссии;"

In [ ]:
from sentence_transformers import SentenceTransformer, util

# -------- Модель эмбеддингов --------
embed_model_name = "deepvk/USER2-base"
embed_model = SentenceTransformer(embed_model_name, device=device)

def cos_sim(model, text1, text2):
  emb1 = model.encode(text1, convert_to_tensor=True, normalize_embeddings=True)
  emb2 = model.encode(text2, convert_to_tensor=True, normalize_embeddings=True)
  return util.cos_sim(emb1, emb2).item()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------- Модель перефразирования --------
paraphrase_model_name = "fyaronskiy/ruT5-large-paraphraser"
para_tokenizer = AutoTokenizer.from_pretrained(paraphrase_model_name)
para_model = AutoModelForSeq2SeqLM.from_pretrained(paraphrase_model_name).to(device)

def generate_paraphrase(text, **kwargs):
    x = get_tokens(text)
    src_len = x.input_ids.shape[1]

    if src_len > 400:
        raise ValueError(f"Input too long: {src_len} tokens.")

    # АДАПТАЦИЯ ДЛЯ КОРОТКИХ ТЕКСТОВ
    if src_len < 30:
        min_len = max(2, int(src_len * 0.7))
        max_len = min(512, int(src_len * 1.5))
        default_temp = 0.95
        default_top_p = 0.92
    else:
        # Для длинных: стандартные параметры
        min_len = max(2, int(src_len * 0.5))
        max_len = min(512, max(min_len + 1, int(src_len * 2.0)))
        default_temp = 1.1
        default_top_p = 0.90

    gen_kwargs = {
        "encoder_no_repeat_ngram_size": 3,
        "max_length": max_len,
        "min_length": min_len,
        "no_repeat_ngram_size": 3,
        "do_sample": True,
        "num_return_sequences": 1,
        "top_k": 50,
        "top_p": kwargs.pop("top_p", default_top_p),
        "temperature": kwargs.pop("temperature", default_temp),
    }
    gen_kwargs.update(kwargs)

    output = para_model.generate(**x, **gen_kwargs)
    return para_tokenizer.decode(output[0], skip_special_tokens=True)

def get_tokens(text):
  return para_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=False,
    ).to(device)

def tokens_len(text):
    return get_tokens(text).input_ids.shape[1]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import util
from razdel import sentenize
import re
from tqdm.auto import tqdm

def generate_best_paraphrase(source_text):
    src_len = tokens_len(source_text)

    is_short = src_len < 70
    n_samples = 7 if is_short else 3
    max_retries = 5 if is_short else 3
    base_temp = 0.95 if is_short else 1.1

    source_emb = embed_model.encode(
        source_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    for retry in range(max_retries):
        if retry > 0:
            if is_short or max_retries - retry == 1:
                temp = base_temp + 0.05 * retry
                candidates = [
                    generate_paraphrase(
                        source_text,
                        temperature=temp,
                        top_k=30,
                        top_p=0.85,
                        repetition_penalty=1.1
                    ) for _ in range(n_samples)
                ]
            else:
                temp = base_temp + 0.1 * retry
                candidates = [
                    generate_paraphrase(source_text, temperature=temp)
                    for _ in range(n_samples)
                ]
        else:
            if is_short:
                candidates = [
                    generate_paraphrase(
                        source_text,
                        temperature=base_temp,
                        top_k=30,
                        top_p=0.85,
                        repetition_penalty=1.1
                    ) for _ in range(n_samples)
                ]
            else:
                candidates = [generate_paraphrase(source_text) for _ in range(n_samples)]

        candidates_embs = embed_model.encode(
            candidates,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        sims = util.cos_sim(source_emb, candidates_embs)[0]
        mask = (sims >= SIM_MIN) & (sims <= SIM_MAX)

        if mask.any():
            idxs = torch.nonzero(mask, as_tuple=False).squeeze(1)
            best_idx_in_mask = torch.argmax(sims[idxs]).item()
            best_idx = idxs[best_idx_in_mask].item()

            best_paraphrase = candidates[best_idx]
            best_sim = sims[best_idx].item()
            return best_paraphrase, best_sim

    best_idx = torch.argmax(sims).item()
    return candidates[best_idx], sims[best_idx].item()


def split_long_sentence(sent: str, max_tokens: int = 100):
    current_len = tokens_len(sent)

    if current_len <= max_tokens:
        return [sent]

    # Попытка разбить по знакам препинания
    parts = re.split(r'(?<=[,;:—-])\s+', sent)

    # Если знаков нет, но предложение длинное — режем по словам
    if len(parts) == 1 and current_len > max_tokens:
        words = sent.split()
        chunks = []
        current = ""

        for w in words:
            candidate = f"{current} {w}".strip() if current else w
            if tokens_len(candidate) <= max_tokens:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                current = w

        if current:
            chunks.append(current)

        return chunks if chunks else [sent]

    # Если есть знаки препинания — собираем по ним
    chunks = []
    current = ""

    for part in parts:
        candidate = f"{current} {part}".strip() if current else part

        if tokens_len(candidate) <= max_tokens:
            current = candidate
        else:
            if current:
                chunks.append(current)

            # Если одна часть слишком длинная — режем по словам
            if tokens_len(part) > max_tokens:
                words = part.split()
                sub_current = ""

                for w in words:
                    sub_candidate = f"{sub_current} {w}".strip() if sub_current else w
                    if tokens_len(sub_candidate) <= max_tokens:
                        sub_current = sub_candidate
                    else:
                        if sub_current:
                            chunks.append(sub_current)
                        sub_current = w

                current = sub_current
            else:
                current = part

    if current:
        chunks.append(current)

    return chunks if chunks else [sent]

def paraphrase(source_text):
    sentences = [s.text for s in sentenize(source_text)]
    paraphrased_sentences = []

    for i, s in enumerate(
        tqdm(sentences, total=len(sentences), desc="Sentences", position=0),
        1
    ):
        subs_sentence = split_long_sentence(s, max_tokens=120)
        paraphrased_subs_sentence = []

        for sub_sentence in tqdm(
            subs_sentence,
            total=len(subs_sentence),
            desc=f"Chunks {i}/{len(sentences)}",
            position=1,
            leave=False
        ):
            para_text, para_sim = generate_best_paraphrase(sub_sentence)
            paraphrased_subs_sentence.append(para_text)

        paraphrased_sentences.append(" ".join(paraphrased_subs_sentence))

    return " ".join(paraphrased_sentences)

# -------- 2. Эксплуатация --------
paraphrased_text = paraphrase(source_text)
cos_score = cos_sim(embed_model, source_text, paraphrased_text)

print(f"source_text len: {len(source_text)}, paraphrased_text len: {len(paraphrased_text)}")
print(f"\nCosine similarity (оригинал ↔ перефразированный текст): {cos_score:.3f}")

Sentences:   0%|          | 0/46 [00:00<?, ?it/s]

Chunks 1/46:   0%|          | 0/1 [00:00<?, ?it/s]

W0510 10:13:49.831000 2164 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Chunks 2/46:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks 3/46:   0%|          | 0/1 [00:00<?, ?it/s]

...

Chunks 43/46:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks 44/46:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks 45/46:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks 46/46:   0%|          | 0/1 [00:00<?, ?it/s]

source_text len: 10783, paraphrased_text len: 9079

Cosine similarity (оригинал ↔ перефразированный текст): 0.931


In [ ]:
import copy
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import util


aug_rows = []

aug_file_path = os.path.join(out_path, "train_paraphrase_partial.csv")
if os.path.exists(aug_file_path):
    df_prev = pd.read_csv(aug_file_path)
    aug_rows = df_prev.to_dict(orient="records")
    print(f"Загружено уже сгенерированных примеров: {len(aug_rows)}")
else:
    print("Генерируем с нуля")


label_to_texts = {
    label: df_small.loc[df_small["label"] == label, "text"].tolist()
    for label in small_labels
}
label_to_embs = {}


for label in tqdm(small_labels, desc="Labels"):
    texts_orig = label_to_texts[label]
    orig_count = len(texts_orig)

    label_aug_rows = [r for r in aug_rows if r["label"] == label]
    label_aug_rows_count = len(label_aug_rows)
    current_count_with_aug = orig_count + label_aug_rows_count
    need = TARGET_PER_CLASS - current_count_with_aug
    print(f"\nLabel: {label} | есть {current_count_with_aug}, нужно добить: {need}")

    if need <= 0:
        print(f"Лейбл {label} уже заполнен")
        continue

    label_embs = label_to_embs.get(label)
    if label_embs is None or label_embs.numel() == 0:
        texts_label = copy.deepcopy(texts_orig)
        # for row in aug_rows:
        #     if row["label"] == label:
        #         texts_label.append(row["text"])
        label_embs = embed_model.encode(
            texts_label,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        label_to_embs[label] = label_embs

    orig_idx = 0
    attempts = 0
    max_attempts = need * 20
    first_success = True

    while need > 0 and attempts < max_attempts:
        attempts += 1

        source_text = texts_orig[orig_idx]
        paraphrased_text = paraphrase(source_text)
        original_cosine_sim = cos_sim(embed_model, source_text, paraphrased_text)

        # схожесть с исходным текстом
        if not (SIM_MIN <= original_cosine_sim <= SIM_MAX):
            print(
                f"Пропуск: сходство с source_text не прошло | "
                f"original_cosine_sim={original_cosine_sim:.4f}, "
                f"ожидалось [{SIM_MIN}, {SIM_MAX}]"
            )

            if first_success:
                first_success = False
            else:
                orig_idx = (orig_idx + 1) % orig_count
                first_success = True

            continue

        new_emb = embed_model.encode(
            paraphrased_text,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        sims = util.cos_sim(new_emb, label_embs)[0]
        max_label_cosine_sim = float(torch.max(sims))

        # схожесть со всеми текстами данного лейбла
        if not (SIM_LABEL_MIN <= max_label_cosine_sim <= SIM_LABEL_MAX):
            print(
                f"Пропуск: сходство с текстами лейбла не прошло | "
                f"max_label_cos={max_label_cosine_sim:.4f}, "
                f"ожидалось [{SIM_LABEL_MIN}, {SIM_LABEL_MAX}]"
            )

            if first_success:
                first_success = False
            else:
                orig_idx = (orig_idx + 1) % orig_count
                first_success = True

            continue

        # добавляем в «корпус» лейбла и его эмбеддингов
        label_embs = torch.cat([label_embs, new_emb.unsqueeze(0)], dim=0)
        label_to_embs[label] = label_embs

        aug_rows.append({
            "label": label,
            "text": paraphrased_text,
            "source_text": source_text,
            "cosine_sim": original_cosine_sim,
            "max_label_cosine_sim": max_label_cosine_sim,
            "augmentation_type": "paraphrase",
        })

        df_aug_partial = pd.DataFrame(aug_rows)
        df_aug_partial.to_csv(aug_file_path, index=False)
        print(f"Сохранено {len(aug_rows)} аугментированных примеров в {aug_file_path}")

        orig_idx = (orig_idx + 1) % orig_count
        need -= 1
        first_success = True


# финальное сохранение аугментаций
df_aug = pd.DataFrame(aug_rows)
df_aug.to_csv(aug_file_path, index=False)
print(f"\nИтого аугментированных примеров: {len(df_aug)}")
print(f"Итоговый файл с аугментацией сохранён в: {aug_file_path}")


# склейка с исходным df
df_full = pd.concat([df, df_aug[["label", "text"]]], ignore_index=True)
final_path = os.path.join(out_path, "train_paraphrase.csv")
df_full.to_csv(final_path, index=False)
print(f"Финальный тренировочный датасет сохранён в: {final_path}")

Загружено уже сгенерированных примеров: 673


Labels:   0%|          | 0/26 [00:00<?, ?it/s]


Label: Проект «Трубопроводный транспорт Ещё одного НГКМ» | есть 40, нужно добить: 0
Лейбл Проект «Трубопроводный транспорт Ещё одного НГКМ» уже заполнен

Label: Блок заместителя генерального директора по строительству | есть 40, нужно добить: 0
Лейбл Блок заместителя генерального директора по строительству уже заполнен

Label: Имущественные вопросы | есть 40, нужно добить: 0
Лейбл Имущественные вопросы уже заполнен

Label: Подразделение по информационным технологиям | есть 40, нужно добить: 0
Лейбл Подразделение по информационным технологиям уже заполнен

Label: Проект «Обустройство объектов Новейшей нейти» | есть 40, нужно добить: 0
Лейбл Проект «Обустройство объектов Новейшей нейти» уже заполнен

Label: Блок исполнительного директора по реализации проекта "Большое месторождение" | есть 40, нужно добить: 0
Лейбл Блок исполнительного директора по реализации проекта "Большое месторождение" уже заполнен

Label: Проект "Обустройство площадных объектов НГКМ Поменбше" | есть 40, нужно доби